# Retrieve & Analyze Colab Manifests from Google Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/investigate-oasis-sheets-aYOlG/notebooks/retrieve-colab-manifests.ipynb)

The `download-all-revisions.ipynb` notebook was previously run and saved
manifests to Google Drive. These manifests contain `content_hash` and
`gc_data_hash` for every revision processed.

This notebook retrieves those manifests and cross-references the GC data
hashes against our known CI artifact hashes (V1-V10) to:

1. Verify which revision numbers match each CI version
2. Find the correct revisions for V2, V3, V6 (currently mismatched)
3. Determine if Method C (direct Sheets URL) produced correct revisions

**Quick and lightweight** — no conversion needed, just read existing manifests.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path
from collections import Counter

DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
assert DRIVE_DIR.exists(), f'Drive directory not found: {DRIVE_DIR}'

print(f'Drive directory: {DRIVE_DIR}')
for f in sorted(DRIVE_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}: {f.stat().st_size:,} bytes')
    elif f.is_dir():
        count = sum(1 for _ in f.iterdir() if _.is_file())
        print(f'  {f.name}/ ({count} files)')

In [ ]:
# Known CI data hashes (Identification block stripped)
# These are the full SHA-256 hashes from verify-gc-against-ci.py
CI_DATA_HASHES = {
    'V1':  'e4e961c1618ff112c3a09eb1',
    'V2':  'c21c9fd6de75cfe5ac6a0161',
    'V3':  '85d713818f2654c256d78a60',
    'V4':  '94aa3b46e274e2dcbb4a987d',
    'V5':  '93a197bc3b741bca51f4106f',
    'V6':  '1ebde1fcadd0f0f949776cfe',
    'V7':  '6504ba8eb96fb8588e40a3f6',
    'V8':  'cfaf508484d0f556e4f41a87',
    'V9':  '5784456984fb5f1239bef093',
    'V10': '5784456984fb5f1239bef093',
}

# Build reverse lookup
CI_HASH_TO_VER = {}
for ver, h in CI_DATA_HASHES.items():
    CI_HASH_TO_VER.setdefault(h, []).append(ver)

print(f'Loaded {len(CI_DATA_HASHES)} CI reference hashes')
print(f'{len(CI_HASH_TO_VER)} unique data states')

In [ ]:
# Analyze manifests
for sheet_key in ['ubl25_library', 'ubl25_documents']:
    mp = DRIVE_DIR / f'manifest-{sheet_key}.json'
    if not mp.exists():
        print(f'\n{sheet_key}: MANIFEST NOT FOUND')
        continue

    m = json.loads(mp.read_text())
    revs = m.get('revisions', [])

    print(f'\n{"="*70}')
    print(f'{sheet_key}')
    print(f'{"="*70}')
    print(f'  Total revisions downloaded: {len(revs)}')
    print(f'  Max revision: {m.get("max_rev", "?")}')
    print(f'  Unique content states: {m.get("unique_states", "?")}')
    print(f'  Converted to GC: {m.get("converted", "?")}')
    print(f'  Conversion errors: {m.get("conversion_errors", 0)}')
    print(f'  Partner sheet: {m.get("partner_sheet", "?")}')
    print(f'  Partner rev: {m.get("partner_rev", "?")}')

    # Content hash distribution
    hash_counts = Counter(r.get('content_hash') for r in revs if r.get('content_hash'))
    print(f'\n  Content hash distribution ({len(hash_counts)} unique):')
    for rank, (h, count) in enumerate(hash_counts.most_common(20), 1):
        matching = sorted([r for r in revs if r.get('content_hash') == h], key=lambda r: r['rev'])
        first, last = matching[0], matching[-1]
        gc_tag = ''
        if first.get('gc_data_hash'):
            gc_tag = f' gc_data={first["gc_data_hash"][:16]}...'
        elif first.get('gc_same_as'):
            gc_tag = f' gc->rev-{first["gc_same_as"]}'
        print(f'    {rank:3d}. {h[:20]}... x{count:4d} (rev {first["rev"]}-{last["rev"]}){gc_tag}')

    # Find CI matches in gc_data_hash
    print(f'\n  CI V1-V10 matches (via gc_data_hash):')
    ci_matches_found = set()
    for rev_entry in revs:
        gc_hash = rev_entry.get('gc_data_hash', '')
        if gc_hash:
            # Try matching against CI hashes (prefix match)
            for ci_ver, ci_hash in CI_DATA_HASHES.items():
                if gc_hash.startswith(ci_hash):
                    print(f'    rev-{rev_entry["rev"]}: gc_data={gc_hash[:20]}... '
                          f'-> CI {ci_ver} !!!')
                    ci_matches_found.add(ci_ver)

    # Which CI versions are NOT matched?
    not_found = set(CI_DATA_HASHES.keys()) - ci_matches_found
    if not_found:
        print(f'\n  CI versions NOT matched: {sorted(not_found)}')
        print(f'  (These may require the partner sheet to also vary,'
              f' or the GC conversion used the latest partner rev)')

    # Conversion errors
    errors = [r for r in revs if r.get('gc_error')]
    if errors:
        print(f'\n  Conversion errors ({len(errors)}):')
        for r in errors[:5]:
            print(f'    rev-{r["rev"]}: {r["gc_error"][:100]}')
        if len(errors) > 5:
            print(f'    ... and {len(errors)-5} more')

In [ ]:
# CRITICAL: The download-all-revisions notebook pairs each sheet with
# the LATEST revision of the partner sheet. This means:
#   - Library scans use doc=2204 (latest) as partner
#   - Documents scans use lib=2005 (latest) as partner
#
# But CI V1-V10 used DIFFERENT partner revisions!
#   - V1: lib=1843 + doc=1793 (not doc=2204!)
#   - V2: lib=??? + doc=1793
#   - V6: lib=??? + doc=2190 (not doc=2204!)
#
# So the gc_data_hash in the manifest WON'T match CI hashes directly,
# because the partner sheet content affects the GC output.
#
# HOWEVER: the content_hash (content.xml of the ODS) IS independent of
# the partner sheet. We can use it to find which revision has the same
# spreadsheet state as the correctly-matched revisions.

print('='*70)
print('IMPORTANT: gc_data_hash matching limitation')
print('='*70)
print()
print('The Colab notebook paired each sheet revision with the LATEST')
print('partner revision. But CI runs used historical partner revisions.')
print()
print('Therefore:')
print('  - gc_data_hash may NOT match CI hashes (different partner)')
print('  - content_hash (content.xml) IS sheet-independent and can be')
print('    used to identify which revision has the right spreadsheet state')
print()
print('Strategy: find the content_hash for known-good revisions (V1,V4,V5)')
print('and use the manifest to find which revision ranges share that hash.')
print('Then scan nearby revisions for the missing V2, V3, V6 matches.')

In [ ]:
# Cross-reference known-good content.xml hashes with manifest data
# Our reference hashes (from work-sheets/revision-ods/content-hashes.json)

KNOWN_CONTENT_HASHES = {
    'ubl25_library': {
        '1843': 'ac8eb8226a3bdceed04fa9ce',  # V1 match
        '1868': '69ed22bf6746fd5a84ccf387',  # V3 data match
        '1999': 'dec0bfa6a9a7f2574aa5d232',  # V5 match
        '2005': '716eacf3ebe60d7622e4c9c4',  # V7-V10 match
    },
    'ubl25_documents': {
        '1793': 'f33306fcfe9227d89bbe8ecb',  # V1 match
        '1803': '7d43056bc165dfe5d95ea6ea',  # V3 mapping (but wrong)
        '1983': '8046f7e9d37de3d0d2e3f835',  # V4 match
        '2190': 'a24df7fa42e69416a8eb9de6',  # V5 match
        '2200': '90eea4287a813eac5fbafa6b',  # V8 match
        '2204': '952a3364f23d6fabeb3cdcac',  # V9 match
    },
}

for sheet_key in ['ubl25_library', 'ubl25_documents']:
    mp = DRIVE_DIR / f'manifest-{sheet_key}.json'
    if not mp.exists():
        continue

    m = json.loads(mp.read_text())
    revs = m.get('revisions', [])

    print(f'\n{"="*70}')
    print(f'{sheet_key}: content_hash cross-reference')
    print(f'{"="*70}')

    # Build lookup: content_hash -> list of rev numbers
    hash_to_revs = {}
    for r in revs:
        h = r.get('content_hash')
        if h:
            hash_to_revs.setdefault(h, []).append(r['rev'])

    # Check each known reference
    for ref_rev, ref_hash in KNOWN_CONTENT_HASHES.get(sheet_key, {}).items():
        # Find matching content_hash in manifest
        matching_revs = None
        for full_hash, rev_list in hash_to_revs.items():
            if full_hash.startswith(ref_hash):
                matching_revs = sorted(rev_list)
                break

        if matching_revs:
            print(f'\n  ref rev-{ref_rev} (content={ref_hash[:16]}...):')
            print(f'    Manifest matches: {len(matching_revs)} revisions')
            print(f'    Range: rev-{matching_revs[0]} to rev-{matching_revs[-1]}')
            if int(ref_rev) in matching_revs:
                print(f'    \u2705 Reference rev-{ref_rev} is in manifest matches')
            else:
                print(f'    \u26a0 Reference rev-{ref_rev} NOT in manifest matches!')
                nearest = min(matching_revs, key=lambda x: abs(x - int(ref_rev)))
                print(f'    Nearest match: rev-{nearest}')
        else:
            print(f'\n  ref rev-{ref_rev} (content={ref_hash[:16]}...): NO MATCH in manifest')
            print(f'    This content_hash was not seen in any Colab-downloaded revision!')
            print(f'    THIS IS A RED FLAG: Method C may have returned different content')

    # Also show ALL unique content states with their revision ranges
    print(f'\n  All unique content states ({len(hash_to_revs)}):')
    for h in sorted(hash_to_revs.keys(), key=lambda x: min(hash_to_revs[x])):
        rev_list = sorted(hash_to_revs[h])
        ref_match = ''
        for ref_rev, ref_hash in KNOWN_CONTENT_HASHES.get(sheet_key, {}).items():
            if h.startswith(ref_hash):
                ref_match = f' <- ref rev-{ref_rev}'
                break
        print(f'    {h[:20]}... x{len(rev_list):4d} '
              f'(rev {rev_list[0]}-{rev_list[-1]}){ref_match}')

In [ ]:
# Save manifests locally for offline analysis
import shutil

local_dir = Path('/content/manifests')
local_dir.mkdir(exist_ok=True)

for sheet_key in ['ubl25_library', 'ubl25_documents']:
    src = DRIVE_DIR / f'manifest-{sheet_key}.json'
    if src.exists():
        dst = local_dir / f'manifest-{sheet_key}.json'
        shutil.copy2(src, dst)
        print(f'Saved: {dst} ({dst.stat().st_size:,} bytes)')

# Also save errors log if present
for sheet_key in ['ubl25_library', 'ubl25_documents']:
    err_log = DRIVE_DIR / f'errors-{sheet_key}.log'
    if err_log.exists():
        dst = local_dir / f'errors-{sheet_key}.log'
        shutil.copy2(err_log, dst)
        print(f'Saved: {dst}')

print(f'\nDownload these files for offline analysis:')
print(f'  from google.colab import files')
print(f'  files.download("/content/manifests/manifest-ubl25_library.json")')
print(f'  files.download("/content/manifests/manifest-ubl25_documents.json")')